# Full-Scale Amazon PRMP Validation

**Predictive Residual Message Passing (PRMP)** — a novel GNN message-passing mechanism where parent nodes predict child representations and aggregate the *residuals* to update themselves.

This notebook trains 4 GNN variants on Amazon Video Games review-rating regression:
- **A: Standard** — vanilla heterogeneous GNN (baseline)
- **B: PRMP** — Predictive Residual Message Passing
- **C: Wide** — parameter-matched wider Standard GNN (controls for extra parameters)
- **D: AuxMLP** — Standard + auxiliary prediction loss without residual subtraction

Key metric: **Cohen's d** effect size between variants, measuring whether PRMP's improvement is statistically meaningful.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
# Version pins are for Colab (Python 3.12); locally we allow compatible versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.15.3',
         'matplotlib==3.10.0')
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
import gc
import json
import math
import os
import time
from collections import defaultdict
from hashlib import md5

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}, NumPy {np.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Data Loading
Load the pre-computed experiment data (75 diverse test examples with predictions from all 4 GNN variants).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter7_full_scale_amaz/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
metadata = data["metadata"]
print(f"Loaded {len(examples)} examples")
print(f"Full-scale experiment: {metadata['dataset']['num_reviews']} reviews, "
      f"{metadata['dataset']['num_products']} products, "
      f"{metadata['dataset']['num_customers']} customers")

## Configuration
Tunable parameters for the demo run. Values are set small for fast execution; original full-scale values are shown in comments.

In [ ]:
# ---- Config (original full-scale values) ----
SEEDS = [42, 123, 456, 789, 1024]
HIDDEN_DIM = 128
NUM_LAYERS = 2
LR = 0.001
WEIGHT_DECAY = 1e-5
PATIENCE = 20
MAX_EPOCHS = 200
GRAD_CLIP = 1.0
TEXT_HASH_DIM = 16

## Feature Reconstruction
Reconstruct feature matrices and graph structure from the loaded examples.

In [ ]:
def reconstruct_data(examples):
    feature_names = list(json.loads(examples[0]["input"]).keys())
    n = len(examples)
    n_features = len(feature_names)

    X_child = np.zeros((n, n_features), dtype=np.float32)
    ratings = np.zeros(n, dtype=np.float32)
    prod_strs = []
    cust_strs = []

    for i, ex in enumerate(examples):
        feats = json.loads(ex["input"])
        for j, fn in enumerate(feature_names):
            X_child[i, j] = feats.get(fn, 0.0)
        ratings[i] = float(ex["output"])
        prod_strs.append(ex["metadata_product_id"])
        cust_strs.append(ex["metadata_customer_id"])

    unique_prods = sorted(set(prod_strs))
    unique_custs = sorted(set(cust_strs))
    product_id_map = {p: i for i, p in enumerate(unique_prods)}
    customer_id_map = {c: i for i, c in enumerate(unique_custs)}

    product_ids = np.array([product_id_map[p] for p in prod_strs], dtype=np.int64)
    customer_ids = np.array([customer_id_map[c] for c in cust_strs], dtype=np.int64)

    n_products = len(unique_prods)
    n_customers = len(unique_custs)

    # Aggregate parent features from child data
    prod_r = {i: [] for i in range(n_products)}
    cust_r = {i: [] for i in range(n_customers)}
    for i in range(n):
        prod_r[product_ids[i]].append(float(ratings[i]))
        cust_r[customer_ids[i]].append(float(ratings[i]))

    product_features = np.zeros((n_products, 5), dtype=np.float32)
    for pid in range(n_products):
        r = prod_r[pid]
        if r:
            product_features[pid] = [np.mean(r), np.std(r), len(r), 0, 0]

    customer_features = np.zeros((n_customers, 5), dtype=np.float32)
    for cid in range(n_customers):
        r = cust_r[cid]
        if r:
            customer_features[cid] = [np.mean(r), np.std(r), len(r), 0, 0]

    product_features = StandardScaler().fit_transform(product_features).astype(np.float32)
    customer_features = StandardScaler().fit_transform(customer_features).astype(np.float32)

    timestamps = np.arange(n, dtype=np.int64)

    return (X_child, product_features, customer_features, ratings,
            product_ids, customer_ids, product_id_map, customer_id_map,
            timestamps, feature_names)

(X_child, prod_feats, cust_feats, ratings,
 prod_ids, cust_ids, prod_map, cust_map,
 timestamps, feature_names) = reconstruct_data(examples)

print(f"Child features: {X_child.shape}")
print(f"Product features: {prod_feats.shape}")
print(f"Customer features: {cust_feats.shape}")
print(f"Ratings range: [{ratings.min():.1f}, {ratings.max():.1f}]")

## Graph Construction
Build a heterogeneous graph with review, product, and customer nodes connected by edges.

In [ ]:
def build_graph(X_child, product_features, customer_features, ratings,
                product_ids, customer_ids, timestamps):
    n_reviews = len(X_child)
    n_products = len(product_features)
    n_customers = len(customer_features)

    review_x = torch.tensor(X_child, dtype=torch.float32)
    product_x = torch.tensor(product_features, dtype=torch.float32)
    customer_x = torch.tensor(customer_features, dtype=torch.float32)
    y = torch.tensor(ratings, dtype=torch.float32)

    prod_ids_t = torch.tensor(product_ids, dtype=torch.long)
    cust_ids_t = torch.tensor(customer_ids, dtype=torch.long)
    rev_ids_t = torch.arange(n_reviews, dtype=torch.long)

    # Temporal split 70/15/15
    sorted_idx = np.argsort(timestamps)
    n_train = int(0.70 * n_reviews)
    n_val = int(0.15 * n_reviews)

    train_mask = torch.zeros(n_reviews, dtype=torch.bool)
    val_mask = torch.zeros(n_reviews, dtype=torch.bool)
    test_mask = torch.zeros(n_reviews, dtype=torch.bool)
    train_mask[sorted_idx[:n_train]] = True
    val_mask[sorted_idx[n_train:n_train + n_val]] = True
    test_mask[sorted_idx[n_train + n_val:]] = True

    print(f"Split: train={train_mask.sum().item()}, "
          f"val={val_mask.sum().item()}, test={test_mask.sum().item()}")

    return {
        "review_x": review_x, "product_x": product_x, "customer_x": customer_x,
        "y": y,
        "prod_to_rev_edge": torch.stack([prod_ids_t, rev_ids_t]),
        "cust_to_rev_edge": torch.stack([cust_ids_t, rev_ids_t]),
        "rev_to_prod_edge": torch.stack([rev_ids_t, prod_ids_t]),
        "rev_to_cust_edge": torch.stack([rev_ids_t, cust_ids_t]),
        "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
        "n_reviews": n_reviews, "n_products": n_products, "n_customers": n_customers,
    }

graph_data = build_graph(X_child, prod_feats, cust_feats, ratings,
                          prod_ids, cust_ids, timestamps)

## Model Definitions
Four GNN variants sharing the same heterogeneous graph structure but differing in their message-passing mechanism.

In [ ]:
def scatter_mean(src, index, dim_size):
    out = torch.zeros(dim_size, src.size(1), device=src.device, dtype=src.dtype)
    count = torch.zeros(dim_size, 1, device=src.device, dtype=src.dtype)
    idx = index.unsqueeze(1).expand_as(src)
    out.scatter_add_(0, idx, src)
    count.scatter_add_(0, index.unsqueeze(1),
                       torch.ones(index.size(0), 1, device=src.device, dtype=src.dtype))
    return out / count.clamp(min=1)


class StandardHeteroLayer(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.msg_p2r = nn.Linear(h, h)
        self.msg_c2r = nn.Linear(h, h)
        self.msg_r2p = nn.Linear(h, h)
        self.msg_r2c = nn.Linear(h, h)
        self.upd_r = nn.Linear(h * 3, h)
        self.upd_p = nn.Linear(h * 2, h)
        self.upd_c = nn.Linear(h * 2, h)

    def forward(self, hr, hp, hc, g):
        nr, nprod, ncust = g["n_reviews"], g["n_products"], g["n_customers"]
        p2r = g["prod_to_rev_edge"]; c2r = g["cust_to_rev_edge"]
        r2p = g["rev_to_prod_edge"]; r2c = g["rev_to_cust_edge"]
        agg_p2r = scatter_mean(self.msg_p2r(hp[p2r[0]]), p2r[1], nr)
        agg_c2r = scatter_mean(self.msg_c2r(hc[c2r[0]]), c2r[1], nr)
        agg_r2p = scatter_mean(self.msg_r2p(hr[r2p[0]]), r2p[1], nprod)
        agg_r2c = scatter_mean(self.msg_r2c(hr[r2c[0]]), r2c[1], ncust)
        new_r = F.relu(self.upd_r(torch.cat([hr, agg_p2r, agg_c2r], 1)))
        new_p = F.relu(self.upd_p(torch.cat([hp, agg_r2p], 1)))
        new_c = F.relu(self.upd_c(torch.cat([hc, agg_r2c], 1)))
        return new_r, new_p, new_c


class StandardHeteroGNN(nn.Module):
    def __init__(self, cd=21, pd_=5, cud=5, h=128, nl=2):
        super().__init__()
        self.proj_r = nn.Linear(cd, h)
        self.proj_p = nn.Linear(pd_, h)
        self.proj_c = nn.Linear(cud, h)
        self.layers = nn.ModuleList([StandardHeteroLayer(h) for _ in range(nl)])
        self.head = nn.Linear(h, 1)
        self.variant = "A_standard"

    def forward(self, g, return_emb=False):
        hr = F.relu(self.proj_r(g["review_x"]))
        hp = F.relu(self.proj_p(g["product_x"]))
        hc = F.relu(self.proj_c(g["customer_x"]))
        for layer in self.layers:
            hr, hp, hc = layer(hr, hp, hc, g)
        out = self.head(hr).squeeze(-1)
        return (out, hr, hp, hc) if return_emb else out


class PRMPHeteroLayer(nn.Module):
    """Predictive Residual Message Passing layer.
    Parent predicts child representation; residual = actual - predicted.
    Parent updates from aggregated residuals (novel mechanism)."""
    def __init__(self, h):
        super().__init__()
        self.pred_p2r = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h))
        self.pred_c2r = nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h))
        self.upd_p = nn.Linear(h * 2, h)
        self.upd_c = nn.Linear(h * 2, h)
        self.msg_p2r = nn.Linear(h, h)
        self.msg_c2r = nn.Linear(h, h)
        self.upd_r = nn.Linear(h * 3, h)

    def forward(self, hr, hp, hc, g):
        nr = g["n_reviews"]; nprod = g["n_products"]; ncust = g["n_customers"]
        p2r = g["prod_to_rev_edge"]; c2r = g["cust_to_rev_edge"]
        pred_rv_p = self.pred_p2r(hp[p2r[0]])
        resid_p = hr[p2r[1]] - pred_rv_p
        agg_res_p = scatter_mean(resid_p, p2r[0], nprod)
        new_p = F.relu(self.upd_p(torch.cat([hp, agg_res_p], 1)))
        pred_rv_c = self.pred_c2r(hc[c2r[0]])
        resid_c = hr[c2r[1]] - pred_rv_c
        agg_res_c = scatter_mean(resid_c, c2r[0], ncust)
        new_c = F.relu(self.upd_c(torch.cat([hc, agg_res_c], 1)))
        agg_p = scatter_mean(self.msg_p2r(hp[p2r[0]]), p2r[1], nr)
        agg_c = scatter_mean(self.msg_c2r(hc[c2r[0]]), c2r[1], nr)
        new_r = F.relu(self.upd_r(torch.cat([hr, agg_p, agg_c], 1)))
        return new_r, new_p, new_c


class PRMPHeteroGNN(nn.Module):
    def __init__(self, cd=21, pd_=5, cud=5, h=128, nl=2):
        super().__init__()
        self.proj_r = nn.Linear(cd, h)
        self.proj_p = nn.Linear(pd_, h)
        self.proj_c = nn.Linear(cud, h)
        self.layers = nn.ModuleList([PRMPHeteroLayer(h) for _ in range(nl)])
        self.head = nn.Linear(h, 1)
        self.variant = "B_prmp"

    def forward(self, g, return_emb=False):
        hr = F.relu(self.proj_r(g["review_x"]))
        hp = F.relu(self.proj_p(g["product_x"]))
        hc = F.relu(self.proj_c(g["customer_x"]))
        for layer in self.layers:
            hr, hp, hc = layer(hr, hp, hc, g)
        out = self.head(hr).squeeze(-1)
        return (out, hr, hp, hc) if return_emb else out


class AuxMLPHeteroGNN(nn.Module):
    def __init__(self, cd=21, pd_=5, cud=5, h=128, nl=2):
        super().__init__()
        self.proj_r = nn.Linear(cd, h)
        self.proj_p = nn.Linear(pd_, h)
        self.proj_c = nn.Linear(cud, h)
        self.layers = nn.ModuleList([StandardHeteroLayer(h) for _ in range(nl)])
        self.aux_p = nn.ModuleList([nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h)) for _ in range(nl)])
        self.aux_c = nn.ModuleList([nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h)) for _ in range(nl)])
        self.head = nn.Linear(h, 1)
        self.variant = "D_aux_mlp"
        self._aux_loss = torch.tensor(0.0)

    def forward(self, g, return_emb=False):
        hr = F.relu(self.proj_r(g["review_x"]))
        hp = F.relu(self.proj_p(g["product_x"]))
        hc = F.relu(self.proj_c(g["customer_x"]))
        p2r = g["prod_to_rev_edge"]; c2r = g["cust_to_rev_edge"]
        aux = torch.tensor(0.0, device=hr.device)
        for i, layer in enumerate(self.layers):
            pred_r_p = self.aux_p[i](hp[p2r[0]])
            aux = aux + F.mse_loss(pred_r_p, hr[p2r[1]].detach())
            pred_r_c = self.aux_c[i](hc[c2r[0]])
            aux = aux + F.mse_loss(pred_r_c, hr[c2r[1]].detach())
            hr, hp, hc = layer(hr, hp, hc, g)
        self._aux_loss = aux
        out = self.head(hr).squeeze(-1)
        return (out, hr, hp, hc) if return_emb else out


def _count_params(m):
    return sum(p.numel() for p in m.parameters())

def make_wide_model(target_params, cd=21, pd_=5, cud=5, nl=2):
    lo, hi = 8, 512
    while lo < hi:
        mid = (lo + hi) // 2
        p = _count_params(StandardHeteroGNN(cd, pd_, cud, mid, nl))
        if p < target_params:
            lo = mid + 1
        else:
            hi = mid
    m = StandardHeteroGNN(cd, pd_, cud, lo, nl)
    m.variant = "C_wide"
    return m, lo

print("Model definitions loaded.")

## Training Loop
Train each GNN variant with early stopping on validation MAE.

In [ ]:
def set_seeds(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

def _to_device(graph_data, device):
    return {k: (v.to(device) if isinstance(v, torch.Tensor) else v)
            for k, v in graph_data.items()}

def train_one_run(model, graph_data, seed, device):
    set_seeds(seed)
    gd = _to_device(graph_data, device)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = float("inf")
    best_state = None
    patience_ctr = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        opt.zero_grad()
        pred = model(gd)
        loss = F.l1_loss(pred[gd["train_mask"]], gd["y"][gd["train_mask"]])
        if hasattr(model, "_aux_loss"):
            loss = loss + 0.1 * model._aux_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

        model.eval()
        with torch.no_grad():
            vp = model(gd)
            vm = F.l1_loss(vp[gd["val_mask"]], gd["y"][gd["val_mask"]]).item()
        if vm < best_val:
            best_val = vm
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        out_tuple = model(gd, return_emb=True)
        pred_all, hr, hp, hc = out_tuple
        pred_np = pred_all.cpu().numpy()
        y_np = gd["y"].cpu().numpy()
        tm = gd["test_mask"].cpu().numpy()

    t_mae = float(np.mean(np.abs(pred_np[tm] - y_np[tm])))
    t_rmse = float(np.sqrt(np.mean((pred_np[tm] - y_np[tm]) ** 2)))
    ss_res = float(np.sum((y_np[tm] - pred_np[tm]) ** 2))
    ss_tot = float(np.sum((y_np[tm] - y_np[tm].mean()) ** 2))
    t_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

    emb = {"review": hr.cpu().numpy(),
           "product": hp.cpu().numpy(),
           "customer": hc.cpu().numpy()}
    model.cpu(); gc.collect()
    return {"test_mae": t_mae, "test_rmse": t_rmse, "test_r2": t_r2,
            "best_val_mae": best_val, "n_epochs": epoch + 1,
            "predictions": pred_np}, emb

print("Training functions ready.")

## Run Training
Train all 4 variants and collect results.

In [ ]:
cd, pd_, cud = X_child.shape[1], prod_feats.shape[1], cust_feats.shape[1]

prmp_params = _count_params(PRMPHeteroGNN(cd, pd_, cud, HIDDEN_DIM, NUM_LAYERS))
std_params = _count_params(StandardHeteroGNN(cd, pd_, cud, HIDDEN_DIM, NUM_LAYERS))
wide_tmp, wide_h = make_wide_model(prmp_params, cd, pd_, cud, NUM_LAYERS)
wide_params = _count_params(wide_tmp); del wide_tmp
aux_params = _count_params(AuxMLPHeteroGNN(cd, pd_, cud, HIDDEN_DIM, NUM_LAYERS))
print(f"Params: Std={std_params}, PRMP={prmp_params}, Wide(h={wide_h})={wide_params}, Aux={aux_params}")

factories = [
    ("A_standard", lambda: StandardHeteroGNN(cd, pd_, cud, HIDDEN_DIM, NUM_LAYERS)),
    ("B_prmp",     lambda: PRMPHeteroGNN(cd, pd_, cud, HIDDEN_DIM, NUM_LAYERS)),
    ("C_wide",     lambda: make_wide_model(prmp_params, cd, pd_, cud, NUM_LAYERS)[0]),
    ("D_aux_mlp",  lambda: AuxMLPHeteroGNN(cd, pd_, cud, HIDDEN_DIM, NUM_LAYERS)),
]

all_results = {}
all_preds = {}

t_start = time.time()
for vname, factory in factories:
    print(f"\nTraining {vname}...")
    vr = []
    vp = []
    for seed in SEEDS:
        model = factory()
        model.variant = vname
        metrics, emb = train_one_run(model, graph_data, seed, DEVICE)
        print(f"  seed={seed}: MAE={metrics['test_mae']:.4f}, RMSE={metrics['test_rmse']:.4f}, "
              f"R2={metrics['test_r2']:.4f}, epochs={metrics['n_epochs']}")
        vr.append(metrics)
        vp.append(metrics["predictions"])
        del model, emb; gc.collect()
    all_results[vname] = vr
    all_preds[vname] = np.mean(vp, axis=0)

print(f"\nTotal training time: {time.time() - t_start:.1f}s")

## Statistical Comparison
Compare demo results alongside the pre-computed full-scale results (50K reviews, 5 seeds, 200 epochs).

In [ ]:
def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    n1, n2 = len(a), len(b)
    v1, v2 = np.var(a, ddof=1), np.var(b, ddof=1)
    ps = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
    return 0.0 if ps < 1e-12 else float((a.mean() - b.mean()) / ps)

def compare_variants(ra, rb):
    ma = [r["test_mae"] for r in ra]
    mb = [r["test_mae"] for r in rb]
    if len(ma) < 2:
        md_val = float(np.mean(ma) - np.mean(mb))
        return {"mean_diff": round(md_val, 6), "cohens_d": 0.0, "p_value": 1.0,
                "mae_a_mean": round(float(np.mean(ma)), 6),
                "mae_b_mean": round(float(np.mean(mb)), 6)}
    t, p = stats.ttest_rel(ma, mb)
    d = cohens_d(ma, mb)
    diff = np.array(ma) - np.array(mb)
    md_val = float(diff.mean())
    se = float(stats.sem(diff)) if len(diff) > 1 else 0.0
    ci = [round(md_val - 1.96 * se, 6), round(md_val + 1.96 * se, 6)]
    return {"mean_diff": round(md_val, 6), "cohens_d": round(d, 4),
            "p_value": round(float(p), 6), "ci_95": ci,
            "mae_a_mean": round(float(np.mean(ma)), 6),
            "mae_b_mean": round(float(np.mean(mb)), 6)}

print("=" * 60)
print("DEMO RUN RESULTS (small-scale)")
print("=" * 60)
for tag, a, b in [("B_vs_A (PRMP vs Standard)", "A_standard", "B_prmp"),
                   ("B_vs_C (PRMP vs Wide)",     "C_wide",     "B_prmp"),
                   ("B_vs_D (PRMP vs AuxMLP)",   "D_aux_mlp",  "B_prmp")]:
    c = compare_variants(all_results[a], all_results[b])
    print(f"  {tag}: MAE_a={c['mae_a_mean']:.4f}, MAE_b={c['mae_b_mean']:.4f}, "
          f"diff={c['mean_diff']:.4f}")

print("\n" + "=" * 60)
print("FULL-SCALE RESULTS (from pre-computed experiment)")
print("=" * 60)
full_comp = metadata["comparisons"]
for tag, key in [("B_vs_A (PRMP vs Standard)", "B_vs_A"),
                  ("B_vs_C (PRMP vs Wide)", "B_vs_C"),
                  ("B_vs_D (PRMP vs AuxMLP)", "B_vs_D")]:
    c = full_comp[key]
    print(f"  {tag}: diff={c['mean_diff']:.4f}, Cohen's d={c['cohens_d']:.4f}, "
          f"p={c['p_value']:.4f}")

print(f"\nKey finding: {metadata['key_finding']}")

## Visualization
Compare MAE across all 4 variants using the full-scale pre-computed results (5 seeds x 200 epochs x 50K reviews).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: MAE per variant (full-scale results) ---
ax = axes[0]
variant_names = ["A_standard", "B_prmp", "C_wide", "D_aux_mlp"]
labels = ["Standard", "PRMP", "Wide", "AuxMLP"]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

mae_means = [metadata["variants"][v]["mae_mean"] for v in variant_names]
mae_stds = [metadata["variants"][v]["mae_std"] for v in variant_names]

bars = ax.bar(labels, mae_means, yerr=mae_stds, color=colors, capsize=5,
              edgecolor="black", linewidth=0.5)
ax.set_ylabel("Test MAE", fontsize=12)
ax.set_title("Full-Scale MAE (50K reviews, 5 seeds)", fontsize=12)
ax.set_ylim(0.34, 0.40)
for bar, val in zip(bars, mae_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{val:.4f}", ha='center', va='bottom', fontsize=10)

# --- Plot 2: Cohen's d effect sizes ---
ax = axes[1]
comparisons = metadata["comparisons"]
comp_labels = ["PRMP\nvs Std", "PRMP\nvs Wide", "PRMP\nvs AuxMLP"]
d_values = [comparisons["B_vs_A"]["cohens_d"],
            comparisons["B_vs_C"]["cohens_d"],
            comparisons["B_vs_D"]["cohens_d"]]
bar_colors = ["#DD8452" if abs(d) > 0.5 else "#AAAAAA" for d in d_values]
bars = ax.bar(comp_labels, d_values, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=-0.5, color='red', linewidth=0.8, linestyle='--', alpha=0.5,
           label='|d|=0.5 (medium)')
ax.axhline(y=0.5, color='red', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_ylabel("Cohen's d", fontsize=12)
ax.set_title("Effect Size Comparisons", fontsize=12)
ax.legend(fontsize=9)
for bar, val in zip(bars, d_values):
    y_pos = bar.get_height() - 0.08 if val < 0 else bar.get_height() + 0.03
    ax.text(bar.get_x() + bar.get_width()/2, y_pos,
            f"d={val:.3f}", ha='center', fontsize=10, fontweight='bold')

# --- Plot 3: Prediction vs Actual scatter (demo run) ---
ax = axes[2]
test_mask = graph_data["test_mask"].numpy()
y_test = ratings[test_mask]
for vname, label, color in zip(["A_standard", "B_prmp"],
                                ["Standard", "PRMP"],
                                ["#4C72B0", "#DD8452"]):
    preds_test = all_preds[vname][test_mask]
    ax.scatter(y_test, preds_test, alpha=0.7, s=40, c=color,
               label=label, edgecolors='white', linewidth=0.3)
lims = [min(y_test.min(), 0), max(y_test.max(), 5)]
ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect')
ax.set_xlabel("Actual Rating", fontsize=12)
ax.set_ylabel("Predicted Rating", fontsize=12)
ax.set_title("Demo: Predictions vs Actual", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("prmp_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved to prmp_results.png")